# Sentinel — Bias Audit Demo
**Legal basis:** ECOA, Fair Housing Act, Title VII, EU AI Act Art.10

Three industry scenarios. Disparate Impact Ratio, Equalized Odds Gap, Demographic Parity Gap.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from sentinel.bias_auditor import BiasAuditor, GroupStats
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

auditor = BiasAuditor()
print('Sentinel Bias Auditor loaded')
print(f'DIR threshold: {BiasAuditor.DIR_THRESHOLD} (EEOC 4/5ths rule)')
print(f'EOG threshold: {BiasAuditor.EOG_THRESHOLD}')
print(f'DPG threshold: {BiasAuditor.DPG_THRESHOLD}')

ModuleNotFoundError: No module named 'seaborn'

## Industry 1 — Loan Approval AI (Bank)

In [2]:
loan_groups = [
    GroupStats('White',           5000, 3900, 3700, 200, 4100),
    GroupStats('Black',           2800, 1708, 1560, 148, 1900),
    GroupStats('Hispanic',        2200, 1276, 1150, 126, 1500),
    GroupStats('Asian',           1800, 1476, 1400,  76, 1600),
    GroupStats('Native American',  400,  196,  180,  16,  220),
]
loan_result = auditor.audit(loan_groups)
print('Loan AI Bias Audit')
print(f'DIR: {loan_result["dir"]} ({"VIOLATION" if loan_result["dir_flag"] else "OK"})')
print(f'EOG: {loan_result["eog"]} ({"VIOLATION" if loan_result["eog_flag"] else "OK"})')
print(f'DPG: {loan_result["dpg"]} ({"VIOLATION" if loan_result["dpg_flag"] else "OK"})')
print(f'Legal risk: {loan_result["legal_risk"]}')
print(f'Compliant: {loan_result["compliant"]}')

NameError: name 'auditor' is not defined

## Industry 2 — Hiring Screening AI

In [3]:
hiring_groups = [
    GroupStats('Male_Under40',   3000, 1980, 1900, 80, 2200),
    GroupStats('Male_Over40',    2000, 1160, 1100, 60, 1400),
    GroupStats('Female_Under40', 2800, 1624, 1550, 74, 1900),
    GroupStats('Female_Over40',  1800,  864,  810, 54, 1100),
]
hiring_result = auditor.audit(hiring_groups)
print('Hiring AI Bias Audit')
print(f'DIR: {hiring_result["dir"]} ({"VIOLATION" if hiring_result["dir_flag"] else "OK"})')
print(f'EOG: {hiring_result["eog"]} ({"VIOLATION" if hiring_result["eog_flag"] else "OK"})')
print(f'DPG: {hiring_result["dpg"]} ({"VIOLATION" if hiring_result["dpg_flag"] else "OK"})')
print(f'Legal risk: {hiring_result["legal_risk"]}')

NameError: name 'auditor' is not defined

## Industry 3 — Medical Triage AI

In [1]:
medical_groups = [
    GroupStats('Private_Insurance', 4000, 3360, 3200, 160, 3600),
    GroupStats('Medicare',          3000, 2310, 2200, 110, 2700),
    GroupStats('Medicaid',          2500, 1750, 1650, 100, 2000),
    GroupStats('Uninsured',         1500,  855,  800,  55, 1050),
]
medical_result = auditor.audit(medical_groups)
print('Medical AI Bias Audit')
print(f'DIR: {medical_result["dir"]} ({"VIOLATION" if medical_result["dir_flag"] else "OK"})')
print(f'EOG: {medical_result["eog"]} ({"VIOLATION" if medical_result["eog_flag"] else "OK"})')
print(f'DPG: {medical_result["dpg"]} ({"VIOLATION" if medical_result["dpg_flag"] else "OK"})')
print(f'Legal risk: {medical_result["legal_risk"]}')

NameError: name 'GroupStats' is not defined

## Comparison Chart — DIR Across All Industries

In [4]:
industries = ['Loan Approval AI', 'Hiring Screening AI', 'Medical Triage AI']
dir_values = [loan_result['dir'], hiring_result['dir'], medical_result['dir']]
dpg_values = [loan_result['dpg'], hiring_result['dpg'], medical_result['dpg']]
eog_values = [loan_result['eog'], hiring_result['eog'], medical_result['eog']]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sentinel Bias Audit — Three Industries', fontsize=15, fontweight='bold', y=1.02)

bar_colors = ['#dc3545' if v < 0.80 else '#28a745' for v in dir_values]
axes[0].bar(industries, dir_values, color=bar_colors, edgecolor='white')
axes[0].axhline(y=0.80, color='red', linestyle='--', linewidth=2, label='EEOC threshold 0.80')
axes[0].set_title('Disparate Impact Ratio\n(higher = more equal)', fontweight='bold')
axes[0].set_ylabel('DIR')
axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=8)
for i, v in enumerate(dir_values):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

dpg_colors = ['#dc3545' if v > 0.05 else '#28a745' for v in dpg_values]
axes[1].bar(industries, dpg_values, color=dpg_colors, edgecolor='white')
axes[1].axhline(y=0.05, color='red', linestyle='--', linewidth=2, label='Threshold 0.05')
axes[1].set_title('Demographic Parity Gap\n(lower = more equal)', fontweight='bold')
axes[1].set_ylabel('DPG')
axes[1].legend(fontsize=8)
for i, v in enumerate(dpg_values):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

eog_colors = ['#dc3545' if v > 0.05 else '#28a745' for v in eog_values]
axes[2].bar(industries, eog_values, color=eog_colors, edgecolor='white')
axes[2].axhline(y=0.05, color='red', linestyle='--', linewidth=2, label='Threshold 0.05')
axes[2].set_title('Equalized Odds Gap\n(lower = more equal)', fontweight='bold')
axes[2].set_ylabel('EOG')
axes[2].legend(fontsize=8)
for i, v in enumerate(eog_values):
    axes[2].text(i, v + 0.002, f'{v:.3f}', ha='center', fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

NameError: name 'loan_result' is not defined

## Positive Outcome Rates by Group — Loan AI

In [3]:
groups = ['White', 'Black', 'Hispanic', 'Asian', 'Native American']
rates  = list(loan_result['positive_rates'].values())
colors = ['#28a745' if r >= 0.80 * max(rates) else '#dc3545' for r in rates]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(groups, rates, color=colors, edgecolor='white')
ax.axvline(x=0.80 * max(rates), color='red', linestyle='--',
           linewidth=2, label=f'4/5ths threshold ({0.80*max(rates):.2f})')
ax.set_title('Loan Approval Rates by Demographic Group', fontsize=13, fontweight='bold')
ax.set_xlabel('Approval Rate')
ax.legend()
for bar, rate in zip(bars, rates):
    ax.text(rate + 0.005, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%}', va='center', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
print(f'Groups below 4/5ths threshold: {[g for g, r in zip(groups, rates) if r < 0.80 * max(rates)]}')

NameError: name 'loan_result' is not defined